# 2026-10-21_7 BC 堆叠小提琴图——NSG1 / PDE10A（stage 版 + age 版，均剔除 70years 与 30years）

**需求与口径**（沿用 2026-10-21_5 MG 版约定）

- 细胞范围：**仅 BC（bipolar cells）**；基因：**NSG1、PDE10A**（按此顺序自上而下）
- **剔除供体**：**70years**（仅 macula/central，无 peripheral 对照；源表提取时已剔除）与 **30years**（BC 仅 peripheral、无 central 对照）——stage 版与 age 版均剔除
- 两个版本：
  1. **stage 版**：X 轴 Early-central; Early-peripheral; Mid-central; Mid-peripheral; Late-central; Late-peripheral（成年页 young-central; young-peripheral; old-central; old-peripheral；young = 21/25/32years，old = 67/75/86years）
  2. **age 版**（`_byAge` 后缀）：X 轴按 donor_age 逐龄 × central/peripheral 交替——胚胎 12 龄（70→165days）；成年 6 龄（21/25/32/67/75/86years）
- **BC 数据特点（空槽位，保留显示）**：胚胎 Early/Mid 阶段 BC 仅存在于 central（Early-peripheral、Mid-peripheral 无细胞）；70days 供体无 BC 细胞（age 版该龄两槽位均空）；胚胎 79–116days 的 peripheral 槽位为空；137days 起central/peripheral 齐全
- 两版均为：胚胎/成年分页（同一 PDF 页 1 胚胎、页 2 成年）；配色 stage 本色 = central、浅色（lighten 0.45）= peripheral（age 版各龄继承其 stage 颜色，无图例、副标题说明）；sharedY（同一基因两页 y 轴一致，上限跨两期全局计算）；空分组经 `scale_x_discrete(drop = FALSE)` 保留槽位
- 仅输出 PDF，不输出 PNG

**数据源（复用每细胞表，不重读大对象）**

- `Data/2026-10-21_4_BC_Cone_AC_gene_percell_expr.rds`（BC/Cone/AC 每细胞 counts+lognorm，7 基因含 NSG1/PDE10A，成年已剔除 70years；载入后取 BC 并剔除 30years）

**输出**

- `Plot/2026-10-21_7_BC_NSG1_PDE10A_stacked_violin_sharedY.pdf`（stage 版，2 页，11×3.5 in）
- `Plot/2026-10-21_7_BC_NSG1_PDE10A_stacked_violin_sharedY_byAge.pdf`（age 版，2 页，14×3.5 in）

In [ ]:
# ---- 1. 环境与参数 ----------------------------------------------------------
suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(ggplot2)
  library(scales)
})
set.seed(1)

DATE_STR <- "2026-10-21_7"
PROJ_DIR <- "/project/work/2026-10-21_violinplot"
DATA_DIR <- file.path(PROJ_DIR, "Data")
PLOT_DIR <- file.path(PROJ_DIR, "Plot")

PERCELL_RDS <- file.path(DATA_DIR, "2026-10-21_4_BC_Cone_AC_gene_percell_expr.rds")
stopifnot(file.exists(PERCELL_RDS))

CELLTYPE <- "BC"
GENES <- c("NSG1", "PDE10A")

# 剔除供体（两版一致）：70years 仅 central（源表已剔除）、30years 仅 peripheral（本脚本剔除）
EXCLUDE_AGES <- c("70years", "30years")

GROUP_LEVELS_EMB <- c("Early-central", "Early-peripheral", "Mid-central",
                      "Mid-peripheral", "Late-central", "Late-peripheral")
GROUP_LEVELS_ADU <- c("young-central", "young-peripheral", "old-central", "old-peripheral")

AGE_ORDER_EMB <- c('70days','79days','87days','91days','100days','103days',
                   '116days','137days','141days','142days','162days','165days')
AGE_ORDER_ADU <- c('21years','25years','32years','67years','75years','86years')

## 静态 age -> stage 映射（age 版配色用；不依赖数据内是否存在该龄细胞，如 70days 无 BC）
AGE_STAGE_EMB <- c(
  '70days' = 'Early',  '79days' = 'Early',  '87days' = 'Early',  '91days' = 'Early',
  '100days' = 'Mid',   '103days' = 'Mid',   '116days' = 'Mid',
  '137days' = 'Late',  '141days' = 'Late',  '142days' = 'Late',
  '162days' = 'Late',  '165days' = 'Late'
)
AGE_STAGE_ADU <- c(
  '21years' = 'young', '25years' = 'young', '32years' = 'young',
  '67years' = 'old',   '75years' = 'old',   '86years' = 'old'
)
stopifnot(identical(unname(AGE_STAGE_EMB[AGE_ORDER_EMB]),
                    c(rep("Early", 4), rep("Mid", 3), rep("Late", 5))),
          identical(unname(AGE_STAGE_ADU[AGE_ORDER_ADU]),
                    c(rep("young", 3), rep("old", 3))))

# ---- 配色（沿用 panelFH 的 stage/region 逻辑；用户提供）-----------------------
cols.stage.full <- c('#59A14F','#EDC948','#B07AA1','#4E79A7','#E15759')  ## 用户指定统一配色
names(cols.stage.full) <- c('Early','Mid','Late','Young','Old')
cols.stage      <- cols.stage.full[c("Early","Mid","Late")]                       ## dev: Early/Mid/Late
cols.adultstage <- setNames(cols.stage.full[c("Young","Old")], c("young","old"))  ## adult: 数据里为小写 young/old
lighten_col <- function(col, f = 0.45) {
  m <- grDevices::col2rgb(col)
  grDevices::rgb(t(m + (255 - m) * f), maxColorValue = 255)
}
## central = 本色, peripheral = 浅色
region_cols <- function(base) {
  v <- c(base, sapply(base, lighten_col))
  names(v) <- c(paste0(names(base), "_central"), paste0(names(base), "_peripheral"))
  v
}

cols.emb.groups <- region_cols(cols.stage)
names(cols.emb.groups) <- gsub("_", "-", names(cols.emb.groups))
cols.adu.groups <- region_cols(cols.adultstage)
names(cols.adu.groups) <- gsub("_", "-", names(cols.adu.groups))

cat("细胞类型:", CELLTYPE, "| 基因（自上而下）:", paste(GENES, collapse = " -> "), "\n")
cat("剔除供体:", paste(EXCLUDE_AGES, collapse = ", "), "（stage 版与 age 版一致）\n")
cat("\nstage 版分组配色：\n")
print(cols.emb.groups[GROUP_LEVELS_EMB])
print(cols.adu.groups[GROUP_LEVELS_ADU])
cat("\nage 版成年龄序（", length(AGE_ORDER_ADU), "龄）:", paste(AGE_ORDER_ADU, collapse = ", "), "\n")

In [ ]:
# ---- 2. 读取 BC 每细胞表，剔除 70/30years，计算 sharedY 上限 -------------------
percell_all <- readRDS(PERCELL_RDS)
stopifnot(all(paste0(GENES, "_lognorm") %in% colnames(percell_all)))
cat("源表总细胞数（BC/Cone/AC，成年已剔除 70years）:", nrow(percell_all), "\n")

bc_all <- percell_all[percell_all$celltype == CELLTYPE, ]
cat(CELLTYPE, "细胞数（源表）:", nrow(bc_all), "\n")
stopifnot(!("70years" %in% bc_all$donor_age))   # 源表提取时已剔除
cat("已确认源表不含 70years\n")

excl_tab <- table(bc_all$donor_age[bc_all$donor_age %in% EXCLUDE_AGES])
cat("本脚本剔除细胞数：\n"); print(excl_tab)
bc <- bc_all[!(bc_all$donor_age %in% EXCLUDE_AGES), ]
bc$period <- factor(as.character(bc$period), levels = c("Embryonic", "Adult"))
emb <- bc[bc$period == "Embryonic", ]
adu <- bc[bc$period == "Adult", ]
cat("\n剔除后 BC 细胞数：胚胎", nrow(emb), "| 成年", nrow(adu), "| 合计", nrow(bc), "\n")

cat("\n各基因检出率（counts > 0 细胞占比）：\n")
cat("[Embryonic] "); print(round(colMeans(emb[paste0(GENES, "_counts")] > 0), 4))
cat("[Adult]     "); print(round(colMeans(adu[paste0(GENES, "_counts")] > 0), 4))

cat("\n[Embryonic] stage 版分组细胞数（注意 Early/Mid-peripheral 无细胞）：\n")
print(table(factor(as.character(emb$stage_region), levels = GROUP_LEVELS_EMB)))
cat("\n[Adult] stage 版分组细胞数：\n")
print(table(factor(as.character(adu$stage_region), levels = GROUP_LEVELS_ADU)))

## age 版分组变量
bc$age_region <- paste0(bc$donor_age, "\n", bc$region)
emb <- bc[bc$period == "Embryonic", ]
adu <- bc[bc$period == "Adult", ]
make_age_levels <- function(ages) {
  unlist(lapply(ages, function(a) c(paste0(a, "\ncentral"), paste0(a, "\nperipheral"))))
}
GROUP_AGE_EMB <- make_age_levels(AGE_ORDER_EMB)
GROUP_AGE_ADU <- make_age_levels(AGE_ORDER_ADU)
cat("\n[Embryonic] age 版分组细胞数（12 龄 × 2）：\n")
print(table(factor(emb$age_region, levels = GROUP_AGE_EMB)))
cat("\n[Adult] age 版分组细胞数（6 龄 × 2）：\n")
print(table(factor(adu$age_region, levels = GROUP_AGE_ADU)))

## sharedY 上限：剔除后跨两期全局最大值；stage 版与 age 版共用同一基准
gene_y_max <- vapply(GENES, function(g) max(bc[[paste0(g, "_lognorm")]]), numeric(1))
cat("\n每基因跨两期全局最大 lognorm（剔除 70/30years 后）：\n"); print(round(gene_y_max, 3))

build_mini_seurat <- function(df, group_var, group_levels) {
  counts_mat <- Matrix::t(as(as.matrix(df[, paste0(GENES, "_counts")]), "dgCMatrix"))
  data_mat   <- Matrix::t(as(as.matrix(df[, paste0(GENES, "_lognorm")]), "dgCMatrix"))
  rownames(counts_mat) <- GENES; colnames(counts_mat) <- df$cell
  rownames(data_mat)   <- GENES; colnames(data_mat)   <- df$cell
  mini_meta <- data.frame(grp = factor(as.character(df[[group_var]]), levels = group_levels),
                          row.names = df$cell)
  colnames(mini_meta) <- group_var
  mini <- CreateSeuratObject(counts = counts_mat, meta.data = mini_meta)
  mini[["RNA"]]$data <- data_mat
  Idents(mini) <- group_var
  mini
}

In [ ]:
# ---- 3. stage 版：构建两期堆叠小提琴（空槽位保留，同一基因两页 y 轴一致）-------
make_stacked_violin_sharedY <- function(df, group_levels, group_cols, title_str) {
  mini <- build_mini_seurat(df, "stage_region", group_levels)

  ## 实际有细胞的分组（胚胎 BC 在 Early/Mid-peripheral 无细胞，保留空槽位）
  present <- group_levels[group_levels %in% unique(as.character(df$stage_region))]
  if (length(present) < length(group_levels)) {
    cat(sprintf("[%s] 空分组（无细胞，保留空槽位）: %s\n", title_str,
                paste(setdiff(group_levels, present), collapse = " | ")))
  }
  blank_df <- data.frame(
    feature    = factor(GENES, levels = GENES),
    ident      = present[1],
    expression = gene_y_max + pmax(0.001 * gene_y_max, 0.01)
  )

  p <- VlnPlot(mini, features = GENES, group.by = "stage_region",
               stack = TRUE, fill.by = "ident",
               cols = group_cols[group_levels],
               pt.size = 0, flip = TRUE) +
    geom_blank(data = blank_df, aes(x = ident, y = expression), inherit.aes = FALSE) +
    scale_x_discrete(drop = FALSE) +
    scale_y_continuous(expand = c(0, 0), limits = c(0, NA), oob = squish,
                       labels = function(x) c(rep("", length(x) - 2), x[length(x) - 1], "")) +
    labs(title = title_str,
         subtitle = sprintf("BC cells: n = %s | young = 21/25/32y, old = 67/75/86y | per-gene y-axis shared between periods",
                            format(nrow(df), big.mark = ","))) +
    theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 13),
          plot.subtitle = element_text(hjust = 0.5, size = 8.5))

  ## 核对：X 轴分组顺序 / 面板基因顺序 / 填充色逐槽位（空槽位 NA 跳过）/ 每面板 y 范围
  bd <- ggplot_build(p)
  bp <- bd$layout$panel_params[[1]]
  cat(sprintf("[%s] X 轴（分组，左->右）: %s\n", title_str,
              paste(bp$x$get_labels(), collapse = " | ")))
  cat(sprintf("[%s] 面板基因顺序（自上而下）: %s\n", title_str,
              paste(levels(factor(p$data$feature, levels = GENES)), collapse = " | ")))
  fill_by_x <- vapply(seq_along(group_levels), function(i) {
    v <- as.character(bd$data[[1]]$fill[bd$data[[1]]$x == i])
    if (length(v)) unique(v)[1] else NA_character_
  }, character(1))
  expected <- unname(as.character(group_cols[group_levels]))
  hit <- !is.na(fill_by_x)
  stopifnot(all(fill_by_x[hit] == expected[hit]), sum(hit) == length(present))
  cat(sprintf("[%s] 填充色逐槽位校验通过（%d/%d 槽位有细胞）\n", title_str,
              sum(hit), length(group_levels)))
  yr <- do.call(rbind, lapply(bd$layout$panel_params, function(pp) pp$y$get_limits()))
  rownames(yr) <- GENES
  cat(sprintf("[%s] 每基因 y 轴范围：\n", title_str)); print(round(yr, 4))
  invisible(list(plot = p, yranges = yr))
}

vln_emb <- make_stacked_violin_sharedY(emb, GROUP_LEVELS_EMB, cols.emb.groups, "Embryonic BC")
vln_adu <- make_stacked_violin_sharedY(adu, GROUP_LEVELS_ADU, cols.adu.groups, "Adult BC")

## 校验：两页每基因 y 轴范围逐一相等
cat("\n[stage 版] 两页 y 轴范围是否一致（逐基因）：\n")
ok <- vapply(GENES, function(g) isTRUE(all.equal(vln_emb$yranges[g, ], vln_adu$yranges[g, ])),
             logical(1))
print(ok); stopifnot(all(ok))
cat("[stage 版] 校验通过：同一基因胚胎页与成年页 y 轴范围完全一致\n")

print(vln_emb$plot)
print(vln_adu$plot)

In [ ]:
# ---- 4. 输出 stage 版 PDF（页 1 胚胎 / 页 2 成年）-----------------------------
pdf_stage <- file.path(PLOT_DIR, paste0(DATE_STR, "_BC_NSG1_PDE10A_stacked_violin_sharedY.pdf"))
pdf(pdf_stage, width = 11, height = 3.5, useDingbats = FALSE)
print(vln_emb$plot)
print(vln_adu$plot)
dev.off()
cat("已输出 stage 版 PDF（2 页）:", basename(pdf_stage), "\n")
cat("文件大小 (MB):", round(file.size(pdf_stage) / 1024^2, 3), "\n")

In [ ]:
# ---- 5. age 版：各龄继承 stage 颜色，构建两期堆叠小提琴 ------------------------
## 颜色：各 age 继承其 stage 颜色（central = 本色，peripheral = 浅色 45%）
## age -> stage 用静态映射（70days 无 BC 细胞，无法从数据反查）
cols_all_stage <- c(cols.stage, cols.adultstage)
make_age_cols <- function(ages, age_stage_map) {
  st <- unname(age_stage_map[ages])
  stopifnot(all(st %in% names(cols_all_stage)))
  base <- cols_all_stage[st]
  v <- unlist(lapply(base, function(b) c(b, lighten_col(b))))
  setNames(v, make_age_levels(ages))
}
cols.age.emb <- make_age_cols(AGE_ORDER_EMB, AGE_STAGE_EMB)
cols.age.adu <- make_age_cols(AGE_ORDER_ADU, AGE_STAGE_ADU)

## 交叉核对：有细胞的龄，其数据内 stage 与静态映射一致
for (a in c(AGE_ORDER_EMB, AGE_ORDER_ADU)) {
  obs <- unique(as.character(bc$stage[bc$donor_age == a]))
  if (length(obs) > 0) stopifnot(identical(obs, c(AGE_STAGE_EMB, AGE_STAGE_ADU)[[a]]))
}
cat("交叉核对通过：有细胞各龄的数据内 stage 与静态映射一致\n")

cat("胚胎 12 龄对应 stage（静态映射；70days 无 BC 细胞）：\n")
print(AGE_STAGE_EMB[AGE_ORDER_EMB])
cat("\n成年 6 龄对应 stage：\n")
print(AGE_STAGE_ADU[AGE_ORDER_ADU])

make_age_violin_page <- function(df, group_levels, group_cols, title_str) {
  mini <- build_mini_seurat(df, "age_region", group_levels)

  present <- group_levels[group_levels %in% unique(as.character(df$age_region))]
  if (length(present) < length(group_levels)) {
    cat(sprintf("[%s] 空分组（无细胞，保留空槽位）: %s\n", title_str,
                paste(group_levels[!group_levels %in% present], collapse = " | ")))
  }
  blank_df <- data.frame(feature = factor(GENES, levels = GENES), ident = present[1],
                         expression = gene_y_max + pmax(0.001 * gene_y_max, 0.01))

  p <- VlnPlot(mini, features = GENES, group.by = "age_region",
               stack = TRUE, fill.by = "ident",
               cols = group_cols[group_levels], pt.size = 0, flip = TRUE) +
    geom_blank(data = blank_df, aes(x = ident, y = expression), inherit.aes = FALSE) +
    scale_x_discrete(drop = FALSE) +
    scale_y_continuous(expand = c(0, 0), limits = c(0, NA), oob = squish,
                       labels = function(x) c(rep("", length(x) - 2), x[length(x) - 1], "")) +
    labs(title = title_str,
         subtitle = sprintf("BC cells: n = %s | fill = stage color (dark central / light peripheral) | per-gene y-axis shared between periods",
                            format(nrow(df), big.mark = ","))) +
    theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 13),
          plot.subtitle = element_text(hjust = 0.5, size = 8.5),
          axis.text.x = element_text(angle = 0, size = 6.8),
          legend.position = "none")

  ## 核对：X 轴槽位数 / 面板基因顺序 / 填充色逐槽位（空槽位 NA 跳过）/ 每面板 y 范围
  bd <- ggplot_build(p)
  bp <- bd$layout$panel_params[[1]]
  x_levels <- bp$x$get_labels()
  cat(sprintf("[%s] X 轴分组数: %d（首个: %s | 末个: %s）\n", title_str,
              length(x_levels), gsub("\n", "-", x_levels[1]), gsub("\n", "-", tail(x_levels, 1))))
  cat(sprintf("[%s] 面板基因顺序（自上而下）: %s\n", title_str,
              paste(levels(factor(p$data$feature, levels = GENES)), collapse = " | ")))
  fill_by_x <- vapply(seq_along(x_levels), function(i) {
    v <- as.character(bd$data[[1]]$fill[bd$data[[1]]$x == i])
    if (length(v)) unique(v)[1] else NA_character_
  }, character(1))
  expected <- unname(as.character(group_cols[x_levels]))
  hit <- !is.na(fill_by_x)
  stopifnot(all(fill_by_x[hit] == expected[hit]), sum(hit) == length(present))
  cat(sprintf("[%s] 填充色逐槽位校验通过（%d/%d 槽位有细胞）\n", title_str,
              sum(hit), length(x_levels)))
  yr <- do.call(rbind, lapply(bd$layout$panel_params, function(pp) pp$y$get_limits()))
  rownames(yr) <- GENES
  cat(sprintf("[%s] 每基因 y 轴范围：\n", title_str)); print(round(yr, 4))
  invisible(list(plot = p, yranges = yr))
}

vln_age_emb <- make_age_violin_page(emb, GROUP_AGE_EMB, cols.age.emb, "Embryonic BC (by age)")
vln_age_adu <- make_age_violin_page(adu, GROUP_AGE_ADU, cols.age.adu, "Adult BC (by age)")

## 校验：两页每基因 y 轴范围逐一相等（并与 stage 版一致）
cat("\n[age 版] 两页 y 轴范围是否一致（逐基因）：\n")
ok <- vapply(GENES, function(g) isTRUE(all.equal(vln_age_emb$yranges[g, ], vln_age_adu$yranges[g, ])),
             logical(1))
print(ok); stopifnot(all(ok))
cat("[age 版] 校验通过：同一基因胚胎页与成年页 y 轴范围完全一致\n")
stopifnot(identical(vln_emb$yranges, vln_age_emb$yranges))
cat("[交叉校验] stage 版与 age 版 y 轴范围一致（同一套细胞与上限基准）\n")

print(vln_age_emb$plot)
print(vln_age_adu$plot)

In [ ]:
# ---- 6. 输出 age 版 PDF（页 1 胚胎 / 页 2 成年）-------------------------------
pdf_age <- file.path(PLOT_DIR, paste0(DATE_STR, "_BC_NSG1_PDE10A_stacked_violin_sharedY_byAge.pdf"))
pdf(pdf_age, width = 14, height = 3.5, useDingbats = FALSE)
print(vln_age_emb$plot)
print(vln_age_adu$plot)
dev.off()
cat("已输出 age 版 PDF（2 页）:", basename(pdf_age), "\n")
cat("文件大小 (MB):", round(file.size(pdf_age) / 1024^2, 3), "\n")

In [ ]:
# ---- 7. 文件清单与 sessionInfo -----------------------------------------------
fi <- data.frame(
  file = c(basename(pdf_stage), basename(pdf_age)),
  size_MB = round(c(file.size(pdf_stage), file.size(pdf_age)) / 1024^2, 3))
print(fi, row.names = FALSE)
cat("\n全部完成。\n\n")
sessionInfo()